# Mandelbrot & Julia — v2: the Dive Studio

Everything in one place. **Run the three cells in order, once**, then use the Studio panel at the bottom:

1. **Setup** — installs the packages (fast when they're already there).
2. **Engine** — the fractal kernel, the math for self-repeating points, and the movie maker.
3. **Studio** — pick a destination from the menu, look around, preview, and film.

**In the Studio**

| You do | It does |
|---|---|
| Pick a *Destination* | flies the viewer there and shows its Julia twin |
| Left-click the picture | zooms in 2× toward the click and remembers the spot |
| Right-click / shift-click | zooms out 2× |
| *✎ Custom* + **🎯 Snap** | finds the exactly self-repeating points nearest your last click |
| **👁 Preview** | 12 frames from along the film, plus a time estimate |
| **🎬 Render film** | writes `movies/<name>_<date>.mp4` next to this notebook |

**Why these spots repeat forever**

* **Misiurewicz points** (spirals, branches, stars): the critical orbit 0 → c → c² + c → … lands on a repelling cycle. The picture repeats every |λ| of zoom, turned by arg λ, where λ is the cycle's multiplier. *Turn the camera with the spiral* cancels the turn, so the pattern holds still while you fall through it.
* **Renormalization points** (minibrots inside minibrots, and Feigenbaum's bulb chain): a whole copy of the structure comes back every ×δ, with no turn.

Float64 math stays sharp to about ×2⁴⁰ (a trillion), which is as deep as these dives go.

*For A.K. Dewdney's "Computer Recreations," Scientific American, August 1985.*

In [ ]:
# 1 — Setup (run once per session)
%pip install -q numba ipympl ipywidgets imageio imageio-ffmpeg pillow

try:                                   # Google Colab only: allow ipympl widgets
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

In [ ]:
# 2 — The engine (run once): fractal kernel, self-repeating points, dive renderer
#
# Everything the Studio (cell 3) needs lives here, so there is nothing else to
# run in between.
#   escape_counts   numba kernel, smooth escape counts, Brent cycle detection
#                   (+inf = inside the set, renders black)
#   Misiurewicz     snap any click to the nearest exactly self-repeating point
#   DESTINATIONS    hand-checked spots of known geometric beauty
#   make_frame      one titled movie frame;  contact_sheet  12-frame preview
#   write_movie     the H.264 mp4

import math
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib as mpl
from numba import njit, prange
from PIL import Image, ImageDraw, ImageFont

SPAN_HOME, CENTER_HOME = 2.9, -0.75 + 0j
FLOAT64_HALVINGS = 40          # 2^40 zoom: still ~20 float64 steps per pixel

# ---------------------------------------------------------------- the kernel
# fastmath without the "no NaN / no inf" assumptions, so the escape tests below
# stay exactly as written.
_FAST = {"nsz", "arcp", "contract", "afn", "reassoc"}

@njit(parallel=True, fastmath=_FAST)
def escape_counts(re, im, max_iter, cre, cim, julia):
    """Smooth escape counts on the grid re x im.  julia=False: Mandelbrot
       (z0 = 0, c = pixel).  julia=True: Julia set of c = cre + i cim.
       Interior: Brent's cycle detection (any period) -- a return closer than
       1/100 pixel, confirmed by one more lap with |(f^p)'| < 1."""
    out = np.empty((im.size, re.size))
    if julia:
        R = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * np.hypot(cre, cim)))
        bail = 16.0 if R < 4.0 else R * R
    else:
        bail = 16.0
    dx = abs(re[1] - re[0]) if re.size > 1 else 1.0
    tol = min(1e-20, max(1e-30, (0.01 * dx) ** 2))
    for j in prange(im.size):
        yi = im[j]
        y2 = yi * yi
        for i in range(re.size):
            xi = re[i]
            if julia:
                zr, zi, cr, ci = xi, yi, cre, cim
            else:
                zr, zi, cr, ci = 0.0, 0.0, xi, yi
                xq = xi - 0.25
                q = xq * xq + y2
                # main cardioid or period-2 disk: inside, no iteration needed
                if q * (q + xq) <= 0.25 * y2 or (xi + 1.0) * (xi + 1.0) + y2 <= 0.0625:
                    out[j, i] = np.inf
                    continue
            z2 = zr * zr + zi * zi
            if z2 > bail:
                out[j, i] = 1.0 - np.log2(np.log2(z2) * 0.5)
                continue
            mu = np.inf
            sr, si = zr, zi              # Brent's saved point
            power, lam = 1, 0
            vr = vi = 0.0
            left = 0
            wm2 = 1.0
            for k in range(1, max_iter + 1):
                zr, zi = zr * zr - zi * zi + cr, 2.0 * zr * zi + ci
                z2 = zr * zr + zi * zi
                if not z2 <= bail:                       # escaped
                    if np.isfinite(z2) and z2 > 1.0:
                        mu = k - np.log2(np.log2(z2) * 0.5) + 1.0
                    else:
                        mu = float(k)
                    break
                if left > 0:                             # verification lap
                    wm2 *= 4.0 * z2
                    left -= 1
                    if left == 0:
                        dr, di = zr - vr, zi - vi
                        if dr * dr + di * di < tol and wm2 < 1.0:
                            break                        # attracting cycle: inside
                        sr, si, power, lam = zr, zi, 1, 0
                    continue
                lam += 1
                dr, di = zr - sr, zi - si
                if dr * dr + di * di < tol:
                    vr, vi, left, wm2 = zr, zi, lam, 1.0
                elif lam == power:
                    sr, si = zr, zi
                    power *= 2
                    lam = 0
            out[j, i] = mu
    return out

def axes(center, span, px):
    re = np.linspace(center.real - span / 2, center.real + span / 2, px)
    im = np.linspace(center.imag - span / 2, center.imag + span / 2, px)
    return re, im

def auto_iter(span):
    return int(min(300 + 60 * max(0.0, math.log2(SPAN_HOME / span)), 50_000))

def mandel(center, span, px, max_iter=None):
    re, im = axes(center, span, px)
    return escape_counts(re, im, max_iter or auto_iter(span), 0.0, 0.0, False)

def julia(c, px, span=None, max_iter=600):
    span = span or max(3.2, 2.6 * (0.5 + math.sqrt(0.25 + abs(c))))
    re, im = axes(0j, span, px)
    return escape_counts(re, im, max_iter, c.real, c.imag, True)

def colorize(mu, cmap="magma", lim=None):
    """mu -> RGB uint8 (row 0 = top).  lim=None: central 99% of escaped pixels."""
    finite = mu[np.isfinite(mu)]
    if lim is None:
        lo, hi = np.percentile(finite, [0.5, 99.5]) if finite.size else (0.0, 1.0)
    else:
        lo, hi = lim
    v = np.clip((mu - lo) / max(hi - lo, 1e-12), 0.0, 1.0)
    rgba = mpl.colormaps[cmap](v, bytes=True)
    rgba[~np.isfinite(mu)] = (0, 0, 0, 255)
    return np.ascontiguousarray(rgba[::-1, :, :3])

# ------------------------------------------------- self-repeating points
# A Misiurewicz point: the critical orbit 0 -> c -> c^2 + c -> ... lands after
# k steps (preperiod) on a repelling cycle of period p.  Around it the set
# repeats every |lam| of zoom, turned by arg(lam), where lam = (f^p)' = the
# product of 2z over the cycle.  They are dense in the boundary.

def _newton_mis(c, k, p, steps=80):
    step = 1.0
    for _ in range(steps):
        z = dz = zk = dzk = 0j
        for n in range(1, k + p + 1):
            dz = 2.0 * z * dz + 1.0
            z = z * z + c
            if n == k:
                zk, dzk = z, dz
            if abs(z) > 1e6:
                return None
        if dz == dzk:
            return None
        step = (z - zk) / (dz - dzk)
        c -= step
        if abs(step) < 1e-16 * max(1.0, abs(c)):
            return c
    return c if abs(step) < 1e-12 else None

def classify(c, K=40, P=16):
    """(preperiod k, period p, lam) of c's critical orbit, or None."""
    zs = [0j]
    for _ in range(K + 2 * P + 2):
        zs.append(zs[-1] ** 2 + c)
    for p in range(1, P + 1):
        for k in range(0, K + 1):
            tol = 1e-9 * (1.0 + abs(zs[k]))
            if abs(zs[k + p] - zs[k]) < tol:
                if abs(zs[k + 2 * p] - zs[k + p]) > tol or k == 0:
                    return None                  # chance near-miss, or a nucleus
                lam = 1.0 + 0j
                for j in range(k, k + p):
                    lam *= 2.0 * zs[j]
                return (k, p, lam) if abs(lam) > 1.0 + 1e-9 else None
    return None

def snap_misiurewicz(c0, radius=None, K=24, P=8, top=6):
    """Self-repeating points near c0, nearest first: [(c, k, p, lam), ...]."""
    found = []
    for p in range(1, P + 1):
        for k in range(2, K + 1):
            c = _newton_mis(complex(c0), k, p)
            if c is None or not (np.isfinite(c.real) and np.isfinite(c.imag)):
                continue
            info = classify(c)
            if info is None or (radius is not None and abs(c - c0) > radius):
                continue
            if any(abs(c - f[0]) < 1e-11 * max(1.0, abs(c)) for f in found):
                continue
            found.append((c, *info))
    found.sort(key=lambda f: abs(f[0] - c0))
    return found[:top]

@dataclass
class Dest:
    """A dive target.  kind = "misiurewicz" (spirals/branches, colors shift by
       p iterations per repeat) or "renorm" (minibrots inside minibrots, escape
       counts multiply by `ratio` per repeat)."""
    name: str
    c: complex
    kind: str
    scale: float                 # zoom factor between repeats
    turn: float = 0.0            # degrees turned per repeat
    p: int = 1
    k: int = 0
    lam: complex = 0j
    ratio: int = 1
    anchor: complex = None       # renorm: opening-frame center (None -> c)
    max_halvings: float = FLOAT64_HALVINGS
    blurb: str = ""

    @classmethod
    def misiurewicz(cls, name, c, blurb=""):
        info = classify(c)
        if info is None:
            raise ValueError(f"{name}: {c} is not a Misiurewicz point")
        k, p, lam = info
        return cls(name, c, "misiurewicz", abs(lam), math.degrees(np.angle(lam)),
                   p=p, k=k, lam=lam, blurb=blurb)

    @property
    def log2_scale(self):
        return math.log2(self.scale)

    def levels(self, halvings):
        return halvings / self.log2_scale

    def summary(self):
        if self.kind == "misiurewicz":
            s = (f"Misiurewicz point · preperiod {self.k}, period {self.p} · "
                 f"λ = {self.lam.real:+.4g}{self.lam.imag:+.4g}i · repeats every "
                 f"×{self.scale:.4g}, turning {self.turn:+.1f}°")
        else:
            s = (f"Renormalization point · periods ×{self.ratio} per level · "
                 f"repeats every ×{self.scale:.5g}, no turn")
        return s + f" · {self.levels(self.max_halvings):.1f} repeats down to ×2^{self.max_halvings:.0f}"

DESTINATIONS = {d.name: d for d in [
    Dest.misiurewicz("Seahorse Valley — the endless spiral",
                     -0.743291890852430 + 0.131240552308798j,
                     "In the valley between the main cardioid and the big period-2 disk. "
                     "It grows only 1.15× per repeat, so the spiral turns and turns."),
    Dest.misiurewicz("Seahorse Valley — double-spiral star",
                     -0.747188946944443 + 0.113383020648328j,
                     "A little lower in the same valley: a starburst of paired spirals, "
                     "a quarter-turn per repeat."),
    Dest.misiurewicz("Elephant Valley — trunk spiral",
                     0.284884537273174 + 0.011121821889582j,
                     "Right of the cardioid, where the 'elephants' march in toward the cusp at c = 1/4."),
    Dest.misiurewicz("Southern spiral valley",
                     -0.774672446935674 - 0.137429292340917j,
                     "Lower rim of Seahorse Valley. λ ≈ 3 with almost no twist — the spirals come straight at you."),
    Dest.misiurewicz("Three-armed branch point M(4,1)",
                     -0.101096363845622 + 0.956286510809142j,
                     "Top of the period-3 bulb, where three filaments meet; each repeat turns ~120°, "
                     "and six-armed stars bloom along the way."),
    Dest.misiurewicz("Upper dendrite fork",
                     -0.175890705973462 + 1.086624831861379j,
                     "Where the top antenna forks; the branches repeat with a 123° twist."),
    Dest.misiurewicz("c = i — Dewdney's dendrite tip",
                     0.0 + 1.0j,
                     "The 1985 classic: 0 → i → −1+i → −i → −1+i → … so λ = 4+4i, 45° per repeat."),
    Dest.misiurewicz("Antenna crossroads −1.8393",
                     -1.839286755214161 + 0.0j,
                     "On the real antenna, left of the airplane minibrot. λ ≈ −3.36: "
                     "every repeat flips the picture end for end."),
    Dest("Airplane cascade — minibrots forever (tripling point)",
         -1.78644025556364 + 0j, "renorm", 55.247, ratio=3, anchor=-0.75 + 0j,
         blurb="The period-3 'airplane' minibrot holds its own airplane (period 9), which holds "
               "another (27) … A full copy of the set returns every ×55.247, centered and upright."),
    Dest("Feigenbaum point — period doubling forever",
         -1.4011551890920506 + 0j, "renorm", 4.669201609, ratio=2, max_halvings=25,
         blurb="The end of the bulb chain 2, 4, 8, 16, … on the real axis. Repeats every "
               "×4.6692 (Feigenbaum's constant). Escape counts double each repeat, so this "
               "dive stops at ×2^25 to keep render times sane."),
]}

# ---------------------------------------------------------------- the dive
_fontdir = Path(mpl.get_data_path()) / "fonts" / "ttf"
_font_cache = {}
def _font(name, size):
    key = (name, max(int(size), 6))
    if key not in _font_cache:
        try:
            _font_cache[key] = ImageFont.truetype(str(_fontdir / name), key[1])
        except Exception:
            _font_cache[key] = ImageFont.load_default()
    return _font_cache[key]

def _fit(dr, text, name, size, width):
    """Largest font <= size that fits text into width."""
    while size > 7 and dr.textlength(text, font=_font(name, size)) > width:
        size -= 1
    return _font(name, size)
_RS = getattr(Image, "Resampling", Image)

class Dive:
    """Constant-speed zoom (in log2) from the whole set down to dest.c."""

    def __init__(self, dest, halvings=None, cmap="magma", corotate=False):
        self.d = dest
        self.halvings = min(halvings or dest.max_halvings, dest.max_halvings)
        self.cmap = cmap
        self.corotate = corotate and dest.kind == "misiurewicz" and abs(dest.turn) > 1e-9
        self.lim = None

    def view(self, depth):
        s = 2.0 ** (-depth)
        a = self.d.c if self.d.anchor is None else self.d.anchor
        return self.d.c + (a - self.d.c) * s, SPAN_HOME * s      # pure zoom about c

    def max_iter(self, depth):
        level = depth / self.d.log2_scale
        if self.d.kind == "renorm":
            return int(min(300 * self.d.ratio ** level, 650_000))
        return int(min(300 + 60 * depth, 50_000))

    def _shift(self, mu, depth):
        """Escape counts mapped so every repeat gets the same colors."""
        level = depth / self.d.log2_scale
        if self.d.kind == "renorm":
            x = np.full(mu.shape, np.inf)
            ok = np.isfinite(mu) & (mu > 0)
            x[ok] = np.log(mu[ok] / self.d.ratio ** level)
            return x
        return mu - self.d.p * level

    def raw(self, depth, px):
        center, span = self.view(depth)
        if self.corotate:                 # render a bigger square, rotate, crop
            px, span = int(math.ceil(px * math.sqrt(2.0))), span * math.sqrt(2.0)
        re, im = axes(center, span, px)
        return escape_counts(re, im, self.max_iter(depth), 0.0, 0.0, False)

    def calibrate(self, px=240):
        """One fixed color range, measured where the repeats have settled in."""
        depth = min(self.halvings, max(2.0 * self.d.log2_scale, 12.0))
        x = self._shift(self.raw(depth, px), depth)
        f = x[np.isfinite(x)]
        lo, hi = np.percentile(f, [0.5, 99.5]) if f.size else (0.0, 1.0)
        self.lim = (lo, hi if hi > lo else lo + 1.0)

    def picture(self, depth, px):
        if self.lim is None:
            self.calibrate()
        rgb = colorize(self._shift(self.raw(depth, px), depth), self.cmap, self.lim)
        img = Image.fromarray(rgb)
        if self.corotate:
            img = img.rotate(self.d.turn * depth / self.d.log2_scale, resample=_RS.BICUBIC)
            o = (img.width - px) // 2
            img = img.crop((o, o, o + px, o + px))
        return img

    def frame(self, kf, N, px=600):
        """Frame kf of N: the picture plus titles, sized for px."""
        depth = self.halvings * kf / max(N - 1, 1)
        level = depth / self.d.log2_scale
        u = px / 600.0                                    # text scales with the frame
        margin = int(round(120 * u))
        S = px + 2 * margin
        canvas = Image.new("RGB", (S, S), (0, 0, 0))
        canvas.paste(self.picture(depth, px), (margin, margin))
        dr = ImageDraw.Draw(canvas)
        dr.rectangle([margin - 1, margin - 1, margin + px, margin + px], outline=(60, 60, 60))
        y0 = int(margin * 0.22)
        title = self.d.name.upper()
        dr.text((margin, y0), title, font=_fit(dr, title, "DejaVuSans.ttf", 18 * u, px),
                fill=(215, 215, 215))
        f2 = _font("DejaVuSans.ttf", 16 * u)
        c = self.d.c
        dr.text((margin, y0 + 26 * u), f"c* = {c.real:+.13f} {c.imag:+.13f}i",
                font=f2, fill=(160, 160, 160))
        tag = (f"|λ| = {self.d.scale:.3g} · {self.d.turn:+.1f}°" if self.d.kind == "misiurewicz"
               else f"δ = {self.d.scale:.5g}")
        dr.text((margin + px - dr.textlength(tag, font=f2), y0 + 26 * u), tag,
                font=f2, fill=(215, 215, 215))
        big = f"× {round(2.0 ** depth):,d}"
        fb = _font("DejaVuSans-Bold.ttf", 46 * u)
        dr.text(((S - dr.textlength(big, font=fb)) / 2, margin + px + 16 * u), big,
                font=fb, fill=(255, 210, 74))
        if self.d.kind == "renorm":
            n = int(round(level))
            what = "the Mandelbrot set" if n == 0 else f"level {n} · period ×{self.d.ratio}^{n}"
        else:
            what = "camera turning with the spiral" if self.corotate else f"turned {self.d.turn * level:+.0f}°"
        sub = (f"repeat {level:5.2f} / {self.d.levels(self.halvings):.1f} · {what} · "
               f"frame {kf + 1}/{N}")
        fs = _fit(dr, sub, "DejaVuSans.ttf", 16 * u, S - 20)
        dr.text(((S - dr.textlength(sub, font=fs)) / 2, margin + px + 80 * u), sub,
                font=fs, fill=(150, 150, 150))
        return canvas

def contact_sheet(dive, N, px=600, tiles=12, inner=220):
    """12 frames spread along the film, rendered small.  Also returns an
       estimate of seconds per frame at full size px."""
    picks = np.linspace(0, N - 1, tiles).round().astype(int)
    cols, gap = 3, 10
    rows = (tiles + cols - 1) // cols
    tile = inner + 2 * int(round(120 * inner / 600.0))
    sheet = Image.new("RGB", (cols * tile + (cols + 1) * gap, rows * tile + (rows + 1) * gap),
                      (12, 12, 12))
    dive.picture(0.0, 16)                            # calibrate colors outside the timer
    t_total = 0.0
    for t, kf in enumerate(picks):
        t0 = time.time()
        img = dive.frame(int(kf), N, px=inner)
        t_total += time.time() - t0
        sheet.paste(img, (gap + (t % cols) * (tile + gap), gap + (t // cols) * (tile + gap)))
    per_frame = t_total / tiles * (px / inner) ** 2 + 0.04   # + drawing/encoding
    return sheet, per_frame

def write_movie(dive, path, seconds=30, fps=30, px=600, progress=None):
    import imageio.v2 as imageio
    N = int(round(seconds * fps))
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    writer = imageio.get_writer(str(path), fps=fps, codec="libx264", quality=8,
                                pixelformat="yuv420p", macro_block_size=1)
    t0 = time.time()
    try:
        for kf in range(N):
            writer.append_data(np.asarray(dive.frame(kf, N, px=px)))
            if progress:
                progress(kf + 1, N, time.time() - t0)
    finally:
        writer.close()
    return path, time.time() - t0

def movie_name(dest):
    slug = "".join(ch if ch.isalnum() else "_" for ch in dest.name.split("—")[0].strip().lower())
    slug = "_".join(filter(None, slug.split("_")))[:40] or "dive"
    return Path("movies") / f"{slug}_{datetime.now():%Y%m%d_%H%M%S}.mp4"

# warm up (compiles the kernel once) and report
_t0 = time.time()
mandel(CENTER_HOME, SPAN_HOME, 8)
print(f"Engine ready ({time.time() - _t0:.1f} s to compile) · "
      f"{len(DESTINATIONS)} destinations · run cell 3 for the Studio.")

In [ ]:
# 3 — The Studio: choose a destination, look around, preview, film
#
#   Destination menu   hand-picked self-repeating spots, or "Custom"
#   Viewer             left-click: zoom in 2× toward the click (and remember it)
#                      right-click or shift-click: zoom out 2×
#   🎯 Snap            finds the exactly self-repeating points nearest your last
#                      click, lists them, and makes the first one the destination
#   Julia twin         the Julia set for the destination's c — deep in the dive,
#                      the Mandelbrot set looks like this set (Tan Lei's theorem)
#   👁 Preview         12 frames from along the film + a time estimate
#   🎬 Render film     writes movies/<name>_<date>.mp4 next to this notebook

%matplotlib widget

import io
import math
import traceback

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, Video, clear_output

CUSTOM = "✎ Custom — click the picture, then 🎯 Snap"

def _png(img):
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return buf.getvalue()

class Studio:
    def __init__(self, view_px=480):
        self.view_px = view_px
        self.center, self.span = CENTER_HOME, SPAN_HOME
        self.dest = None
        self.pick = None
        self.custom = None
        self.busy = False
        self._quiet = False
        self._build()
        self._on_dest({"new": self.dest_dd.value})     # the menu starts on the first spot

    # ------------------------------------------------------------ layout
    def _build(self):
        wide = W.Layout(width="760px")
        self.dest_dd = W.Dropdown(options=list(DESTINATIONS) + [CUSTOM], description="Destination",
                                  layout=wide, style={"description_width": "90px"})
        self.dest_dd.observe(self._on_dest, "value")
        self.info = W.HTML(layout=wide)

        self.view_out = W.Output()
        with self.view_out:
            self.fig, self.ax = plt.subplots(figsize=(5.0, 5.0))
        self.fig.subplots_adjust(0, 0, 1, 1)
        self.ax.set_axis_off()
        for attr, val in (("toolbar_visible", False), ("header_visible", False),
                          ("footer_visible", False), ("resizable", False)):
            try:
                setattr(self.fig.canvas, attr, val)
            except Exception:
                pass
        self.fig.canvas.mpl_connect("button_press_event", self._on_click)
        self.imshow = None
        self.marks = []

        def button(label, fn, tip=""):
            b = W.Button(description=label, tooltip=tip, layout=W.Layout(width="auto"))
            b.on_click(lambda _: self._guard(fn))
            return b
        self.btn_home = button("⌂ Whole set", self.go_home)
        self.btn_goto = button("◎ Go to destination", self.go_dest)
        self.btn_out = button("− Zoom out", lambda: self.zoom(self.center, 2.0))
        self.view_status = W.HTML()

        self.julia_img = W.Image(format="png", layout=W.Layout(width="240px", height="240px"))
        self.julia_cap = W.HTML(layout=W.Layout(width="250px"))
        self.btn_snap = button("🎯 Snap to nearest repeating point", self.snap,
                               "Uses your last click (or the view center)")
        self.btn_snap.layout.width = "250px"
        self.cand_dd = W.Dropdown(options=[], layout=W.Layout(width="250px"), disabled=True)
        self.cand_dd.observe(self._on_cand, "value")
        self.pick_html = W.HTML(layout=W.Layout(width="250px"))

        self.seconds = W.IntSlider(value=30, min=10, max=120, step=5, description="Length (s)")
        self.fps = W.Dropdown(options=[24, 30, 60], value=30, description="fps",
                              layout=W.Layout(width="150px"))
        self.size = W.Dropdown(options=[("480 px", 480), ("600 px", 600), ("720 px", 720),
                                        ("900 px", 900)], value=600, description="Size",
                               layout=W.Layout(width="170px"))
        self.depth = W.FloatSlider(value=40, min=8, max=40, step=1, description="Depth ×2^",
                                   readout_format=".0f")
        self.cmap = W.Dropdown(options=["magma", "inferno", "twilight_shifted", "viridis",
                                        "cividis", "hot"], value="magma", description="Colors",
                               layout=W.Layout(width="230px"))
        self.cmap.observe(lambda _: self._guard(self.render_view), "value")
        self.corot = W.Checkbox(value=False, description="Turn the camera with the spiral", indent=False)
        self.embed = W.Checkbox(value=True, description="Show the film here (embeds the mp4)", indent=False)

        self.btn_prev = button("👁 Preview", self.preview)
        self.btn_film = button("🎬 Render film", self.film)
        self.btn_film.button_style = "warning"
        self.prog = W.FloatProgress(value=0, min=0, max=100, layout=W.Layout(width="260px"))
        self.stat = W.HTML()
        self.prev_img = W.Image(format="png", layout=W.Layout(max_width="760px"))
        self.vid_out = W.Output()

        viewer = W.VBox([self.view_out, W.HBox([self.btn_home, self.btn_goto, self.btn_out]),
                         self.view_status])
        side = W.VBox([W.HTML("<b>Julia twin</b>"), self.julia_img, self.julia_cap,
                       W.HTML("<hr style='margin:6px 0'>"), self.btn_snap, self.cand_dd,
                       self.pick_html], layout=W.Layout(margin="0 0 0 14px"))
        movie = W.VBox([W.HTML("<b>Film</b>"),
                        W.HBox([self.seconds, self.fps, self.size]),
                        W.HBox([self.depth, self.cmap]),
                        W.HBox([self.corot, self.embed]),
                        W.HBox([self.btn_prev, self.btn_film, self.prog]), self.stat])
        self.ui = W.VBox([self.dest_dd, self.info, W.HBox([viewer, side]),
                          W.HTML("<hr style='margin:8px 0'>"), movie, self.prev_img, self.vid_out])
        self.buttons = [self.btn_home, self.btn_goto, self.btn_out, self.btn_snap,
                        self.btn_prev, self.btn_film]

    # ------------------------------------------------------------ helpers
    def _run(self, fn):
        """Run fn now if we're already inside an action, else as a new action."""
        if self.busy:
            fn()
        else:
            self._guard(fn)

    def _guard(self, fn):
        if self.busy:
            return
        self.busy = True
        for b in self.buttons:
            b.disabled = True
        try:
            fn()
        except Exception:
            self.stat.value = f"<pre style='color:#c33'>{traceback.format_exc()}</pre>"
        finally:
            self.busy = False
            for b in self.buttons:
                b.disabled = False
            self.btn_film.disabled = self.btn_prev.disabled = self.dest is None

    def _set_dest(self, dest):
        self.dest = dest
        if dest is None:
            self.info.value = ("<i>Custom: zoom in by left-clicking on a spot you like on the "
                               "fringe, then press 🎯 Snap.</i>")
            self.julia_img.value = b""
            self.julia_cap.value = ""
            self.btn_film.disabled = self.btn_prev.disabled = True
            return
        self.info.value = (f"<div style='line-height:1.35'>{dest.blurb}<br>"
                           f"<small style='color:#888'>{dest.summary()}</small></div>")
        self.depth.max = dest.max_halvings
        self.depth.value = dest.max_halvings
        spiral = dest.kind == "misiurewicz" and abs(dest.turn) > 1e-9
        self.corot.disabled = not spiral
        if not spiral:
            self.corot.value = False
        self.btn_film.disabled = self.btn_prev.disabled = False
        self.julia_img.value = _png(Image.fromarray(colorize(julia(dest.c, 240), "magma")))
        self.julia_cap.value = ("<small>Julia set of c*. Zoomed far enough in at c*, the "
                                "Mandelbrot set looks like this set near c* (Tan Lei).</small>"
                                if dest.kind == "misiurewicz" else
                                "<small>Julia set of c* — the dynamics behind the cascade.</small>")

    # ------------------------------------------------------------ viewer
    def render_view(self):
        mu = mandel(self.center, self.span, self.view_px)
        rgb = colorize(mu, self.cmap.value)
        h = self.span / 2
        ext = (self.center.real - h, self.center.real + h, self.center.imag - h, self.center.imag + h)
        if self.imshow is None:
            self.imshow = self.ax.imshow(rgb, extent=ext, origin="upper", interpolation="bilinear")
        else:
            self.imshow.set_data(rgb)
            self.imshow.set_extent(ext)
        self.ax.set_xlim(ext[0], ext[1]); self.ax.set_ylim(ext[2], ext[3])
        for m in self.marks:
            m.remove()
        self.marks = []
        def inside(z):
            return abs(z.real - self.center.real) < h and abs(z.imag - self.center.imag) < h
        if self.dest is not None and inside(self.dest.c):
            self.marks.append(self.ax.scatter([self.dest.c.real], [self.dest.c.imag], s=220,
                                              facecolors="none", edgecolors="gold", linewidths=2))
        if self.pick is not None and inside(self.pick):
            self.marks += self.ax.plot([self.pick.real], [self.pick.imag], "+", color="white",
                                       ms=14, mew=2)
        self.fig.canvas.draw_idle()
        self.view_status.value = (f"<small>center {self.center.real:+.12f} {self.center.imag:+.12f}i · "
                                  f"zoom ×{SPAN_HOME / self.span:,.0f}</small>")

    def zoom(self, pt, factor):
        new_span = min(max(self.span * factor, SPAN_HOME / 2 ** FLOAT64_HALVINGS), 4.0)
        f = new_span / self.span
        self.center = pt + (self.center - pt) * f        # keep the click point fixed
        self.span = new_span
        self.render_view()

    def go_home(self):
        self.center, self.span = CENTER_HOME, SPAN_HOME
        self.render_view()

    def go_dest(self):
        if self.dest is None:
            return
        n = math.ceil(10.0 / self.dest.log2_scale)       # whole repeats, about ×2^10 in
        self.center, self.span = Dive(self.dest).view(min(n * self.dest.log2_scale,
                                                          self.dest.max_halvings))
        self.render_view()

    def _on_click(self, ev):
        if self.busy or ev.inaxes is not self.ax or ev.xdata is None:
            return
        pt = complex(ev.xdata, ev.ydata)
        out = ev.button == 3 or "shift" in (ev.key or "")
        if not out and ev.button == 1:
            self.pick = pt
            self.pick_html.value = (f"<small>last click: {pt.real:+.10f} {pt.imag:+.10f}i</small>")
        self._guard(lambda: self.zoom(pt, 2.0 if out else 0.5))

    # ------------------------------------------------------------ choosing
    def _on_dest(self, change):
        if self._quiet:
            return
        name = change["new"]
        if name == CUSTOM:
            self._set_dest(self.custom)
            self.cand_dd.disabled = not self.cand_dd.options
            self._guard(self.render_view)
        else:
            self._set_dest(DESTINATIONS[name])
            self._guard(self.go_dest)

    def snap(self):
        c0 = self.pick if self.pick is not None else self.center
        self.pick_html.value = "<small>searching preperiods 2–24, periods 1–8 …</small>"
        cands = snap_misiurewicz(c0, radius=self.span) or snap_misiurewicz(c0)
        if not cands:
            self.pick_html.value = ("<small style='color:#c33'>No self-repeating point found "
                                    "there — click right on the fringe and try again.</small>")
            return
        opts = []
        for i, (c, k, p, lam) in enumerate(cands):
            slow = " · very slow" if abs(lam) < 1.05 else ""
            opts.append((f"{i + 1}. ×{abs(lam):.3g} per repeat, {math.degrees(np.angle(lam)):+.0f}° · "
                         f"k={k}, p={p}{slow}", i))
        self._cands = cands
        self._quiet = True
        self.dest_dd.value = CUSTOM
        self.cand_dd.options = opts
        self.cand_dd.disabled = False
        self.cand_dd.value = None
        self._quiet = False
        self.cand_dd.value = 0           # -> _on_cand
        self.pick_html.value = (f"<small>{len(cands)} found near your click; gold ring = chosen. "
                                f"'very slow' = barely repeats (looks like a plain zoom).</small>")

    def _on_cand(self, change):
        if self._quiet or change["new"] is None:
            return
        c, k, p, lam = self._cands[change["new"]]
        self.custom = Dest.misiurewicz(f"Your point (preperiod {k}, period {p})", c,
                                       f"Snapped from your click to c* = {c.real:+.12f} {c.imag:+.12f}i.")
        self._set_dest(self.custom)
        self._run(self.render_view)

    # ------------------------------------------------------------ filming
    def _dive(self):
        return Dive(self.dest, halvings=self.depth.value, cmap=self.cmap.value,
                    corotate=self.corot.value)

    def preview(self):
        N = self.seconds.value * self.fps.value
        self.stat.value = "rendering 12 preview frames …"
        sheet, per_frame = contact_sheet(self._dive(), N, px=self.size.value)
        self.prev_img.value = _png(sheet)
        est = per_frame * N
        self.stat.value = (f"Preview ready · the film is {N} frames · rough estimate "
                           f"{est / 60:.1f} min (deep frames are the slow ones)")

    def film(self):
        N = self.seconds.value * self.fps.value
        path = movie_name(self.dest)
        self.prog.value = 0
        self.prog.bar_style = ""
        def progress(k, n, el):
            if k % 5 == 0 or k == n:
                self.prog.value = 100.0 * k / n
                self.stat.value = (f"frame {k}/{n} · {el:,.0f} s elapsed · "
                                   f"~{el / k * (n - k) / 60:.1f} min left (at the current pace)")
        self.stat.value = f"rendering {N} frames → {path} …"
        path, el = write_movie(self._dive(), path, seconds=self.seconds.value,
                               fps=self.fps.value, px=self.size.value, progress=progress)
        self.prog.bar_style = "success"
        mb = path.stat().st_size / 1e6
        self.stat.value = f"✅ saved <code>{path}</code> · {mb:.1f} MB · {el / 60:.1f} min"
        with self.vid_out:
            clear_output()
            if self.embed.value:
                display(Video(str(path), embed=True, width=640))

if "DESTINATIONS" not in globals():
    print("Run cell 2 (the engine) first.")
else:
    studio = Studio()
    display(studio.ui)